# Preflight-only check: does the real run's config actually fit this session's disk, and is the HF token reachable? CPU, no GPU, done in well under a minute.

In [ ]:
import subprocess
r = subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/raut7218/vaani-sed-query.git", "/kaggle/working/query"],
                   capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
assert r.returncode == 0, "clone failed"


In [ ]:
!pip -q install huggingface_hub 2>&1 | tail -3


In [ ]:
import subprocess, os
os.chdir("/kaggle/working/query")
r = subprocess.run(["python", "scripts/preflight.py", "--work", "/kaggle/working",
                    "--max-shards", "0", "--n-synthetic", "20000",
                    "--n-pretrain", "60000", "--clip-len", "8"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)
print("EXIT CODE:", r.returncode)


## Disk-speed check

The actual bug wasn't disk exhaustion itself - it was that a checkpoint write got *slow* (not fast, not failing) once free space got close to the guard's margin, and that slowness silently blocked a DDP barrier for 48+ minutes with zero output. This times a real `save_atomic()` write of a realistically-sized checkpoint with exactly the guard's margin free, to confirm the margin is actually enough - not just enough by arithmetic.

In [ ]:
import shutil, sys, time, torch
sys.path.insert(0, "/kaggle/working/query")
from src.train.train import save_atomic

WORK = "/kaggle/working"
MARGIN_GB = 5.0
free = shutil.disk_usage(WORK).free / 1024**3
filler_gb = max(0.0, free - MARGIN_GB)
filler = os.path.join(WORK, "_filler.bin")
print("[disk-speed] %.1f GB free -> filling %.1f GB to leave the %.1f GB margin"
      % (free, filler_gb, MARGIN_GB))
with open(filler, "wb") as f:
    f.truncate(int(filler_gb * 1024**3))

# ~475 MB: matches state.pt+opt+scaler+ema for the real 23.7M-param model.
dummy = {"model": {"w%d" % i: torch.zeros(2_000_000) for i in range(60)}}
t0 = time.time()
save_atomic(dummy, os.path.join(WORK, "_dummy_ckpt.pt"))
elapsed = time.time() - t0
print("[disk-speed] wrote a ~475 MB checkpoint in %.1fs with %.1f GB free"
      % (elapsed, shutil.disk_usage(WORK).free / 1024**3))
if elapsed > 10:
    print("[disk-speed] SLOW - this is the exact mechanism that hung a real run; "
          "raise the guard margin further before trusting a real run.")
else:
    print("[disk-speed] OK - fast enough that this will not stall a DDP barrier.")

os.remove(filler)
os.remove(os.path.join(WORK, "_dummy_ckpt.pt"))
